# **MoE NN Dataset**
---

In [5]:
# ---------------------------------------------------
# Paths management
# ---------------------------------------------------
import os

# Project path
PROJECT_PATH = os.path.dirname(os.path.dirname(os.getcwd()))

if os.path.exists(PROJECT_PATH):
    print(f"PROJECT_PATH = {PROJECT_PATH}")
else:
    print(f"WARNING! PROJECT_PATH not found! = {PROJECT_PATH}")
    
# Data path
DATA_PATH = PROJECT_PATH + "/Data"
if os.path.exists(DATA_PATH):
    print(f"DATA_PATH = {DATA_PATH}")
else:
    print(f"WARNING! DATA_PATH not found! = {DATA_PATH}")

OUTPUT_DIR = DATA_PATH


# ---------------------------------------------------
# Used paths
# ---------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

PROJECT_PATH = /home/mambo/Documents/Aggregation-of-experts-for-day-ahead-time-series-forecasting
DATA_PATH = /home/mambo/Documents/Aggregation-of-experts-for-day-ahead-time-series-forecasting/Data


In [6]:
# ---------------------------------------------------
# Experts dataset
# ---------------------------------------------------
df_experts = pd.read_csv(DATA_PATH + "/predictions_experts.csv", sep=";")

# Converts date to time format
df_experts["Date_Heure"] = pd.to_datetime(df_experts["Date_Heure"], errors="coerce")

# Keep only specified data
cols_to_keep = ["Date_Heure", "y_true", "Ridge_Global", "RandomForest_Global", "LGBM_Global"]
df_experts_clean = df_experts[cols_to_keep]

# Create a dictionary mapping old names to new names
new_names = {
    "y_true": "y_true",
    "Ridge_Global": "ridge",
    "RandomForest_Global": "randomforest",
    "LGBM_Global": "lgbm"
}

# Apply the rename function
# Inplace=True modifies the existing dataframe directly without needing to reassign it
df_experts_clean.rename(columns=new_names, inplace=True)


print(f"Start date: {df_experts["Date_Heure"].min()}")
print(f"Stop date: {df_experts["Date_Heure"].max()}")
print(f"Samples : {len(df_experts["Date_Heure"])}")
#df_experts_clean.info()
df_experts_clean.head()

Start date: 2024-10-03 00:00:00
Stop date: 2025-11-23 23:00:00
Samples : 10008


/tmp/ipykernel_8004/1834518497.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_experts_clean.rename(columns=new_names, inplace=True)


,Date_Heure,y_true,ridge,randomforest,lgbm
0,2024-10-03 00:00:00,1237.84,1249.726980,1164.385482,1045.849256
1,2024-10-03 01:00:00,1105.38,1068.422459,902.434285,790.083609
2,2024-10-03 02:00:00,1097.06,1013.409821,766.710598,714.894714
3,2024-10-03 03:00:00,970.68,918.601422,699.701733,713.470081
4,2024-10-03 04:00:00,1188.70,790.645030,627.655777,585.931413


In [7]:
# ---------------------------------------------------
# Features dataset
# ---------------------------------------------------
df_feat = pd.read_csv(DATA_PATH + "/data_engineering_belgique.csv")

# Converts date to time format
df_feat["Date_Heure"] = pd.to_datetime(df_feat["Date_Heure"], errors="coerce")
print(f"Start date: {df_feat["Date_Heure"].min()}")
print(f"Stop date: {df_feat["Date_Heure"].max()}")
print(f"Samples : {len(df_feat["Date_Heure"])}")
df_feat.info()
#df_feat.head()

Start date: 2018-01-02 00:00:00
Stop date: 2025-11-23 23:00:00
Samples : 68877
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68877 entries, 0 to 68876
Data columns (total 39 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date_Heure                68877 non-null  datetime64[ns]
 1   Date_Heure.1              68877 non-null  object        
 2   Eolien_MW                 68877 non-null  float64       
 3   Offshore/onshore          68877 non-null  object        
 4   Region                    68877 non-null  object        
 5   speed_longitudinale_100m  68877 non-null  float64       
 6   speed_latitudinale_100m   68877 non-null  float64       
 7   speed_longitudinale_10m   68877 non-null  float64       
 8   speed_latitudinale_10m    68877 non-null  float64       
 9   2m_temperature            68877 non-null  float64       
 10  mean_sea_level_pressure   68877 non-null  float64       
 11  s

In [8]:
# ---------------------------------------------------
# Merge based on "Date_et_Heure"
# ---------------------------------------------------
features = [
    "Date_Heure",
    "Wind_Norm",
    "Wind_Norm_Cubes",
    "wind_cv_3h",
    "Wind_Dir_Meteo_sin",
    "Wind_Dir_Meteo_cos",
    "Air_density",
    "Hour_sin",
    "Hour_cos",
    "Month_sin",
    "Month_cos"
]

# Merge based on "Date_Heure"
df_merged = df_experts_clean.merge(
    df_feat[features],
    on="Date_Heure",
    how="left"
)

df_merged.to_csv(f"{OUTPUT_DIR}/experts_and_features_for_moe_nn.csv")

print(f"Start date: {df_merged["Date_Heure"].min()}")
print(f"Stop date: {df_merged["Date_Heure"].max()}")
print(f"Samples : {len(df_merged["Date_Heure"])}")
df_merged.info()
df_merged.head()

Start date: 2024-10-03 00:00:00
Stop date: 2025-11-23 23:00:00
Samples : 10008
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10008 entries, 0 to 10007
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Date_Heure          10008 non-null  datetime64[ns]
 1   y_true              10008 non-null  float64       
 2   ridge               10008 non-null  float64       
 3   randomforest        10008 non-null  float64       
 4   lgbm                10008 non-null  float64       
 5   Wind_Norm           10008 non-null  float64       
 6   Wind_Norm_Cubes     10008 non-null  float64       
 7   wind_cv_3h          10008 non-null  float64       
 8   Wind_Dir_Meteo_sin  10008 non-null  float64       
 9   Wind_Dir_Meteo_cos  10008 non-null  float64       
 10  Air_density         10008 non-null  float64       
 11  Hour_sin            10008 non-null  float64       
 12  Hour_cos            100

,Date_Heure,y_true,ridge,randomforest,lgbm,Wind_Norm,Wind_Norm_Cubes,wind_cv_3h,Wind_Dir_Meteo_sin,Wind_Dir_Meteo_cos,Air_density,Hour_sin,Hour_cos,Month_sin,Month_cos
0,2024-10-03 00:00:00,1237.84,1249.726980,1164.385482,1045.849256,10.967685,1319.304126,0.013732,0.353894,0.935286,1.238500,0.000000,1.000000,-0.866025,0.5
1,2024-10-03 01:00:00,1105.38,1068.422459,902.434285,790.083609,10.439776,1137.819917,0.035596,0.360263,0.932851,1.239087,0.258819,0.965926,-0.866025,0.5
2,2024-10-03 02:00:00,1097.06,1013.409821,766.710598,714.894714,10.232496,1071.382915,0.035942,0.359241,0.933245,1.239988,0.500000,0.866025,-0.866025,0.5
3,2024-10-03 03:00:00,970.68,918.601422,699.701733,713.470081,9.899611,970.184501,0.026741,0.353904,0.935282,1.241176,0.707107,0.707107,-0.866025,0.5
4,2024-10-03 04:00:00,1188.70,790.645030,627.655777,585.931413,9.421683,836.344942,0.041372,0.372352,0.928092,1.242842,0.866025,0.500000,-0.866025,0.5
